In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
#!pip install -q pmdarima
#!pip install -q tbats
!pip install -q sktime

In [ ]:
import sys
import logging
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sktime.utils.plotting import plot_series
from sktime.forecasting.base import ForecastingHorizon
from sktime.forecasting.exp_smoothing import ExponentialSmoothing
from sktime.forecasting.sarimax import SARIMAX
from sktime.forecasting.var import VAR
from sktime.forecasting.varmax import VARMAX
from sktime.forecasting.vecm import VECM

from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)
warnings.simplefilter("ignore", RuntimeWarning)
pd.options.mode.chained_assignment = None

In [ ]:
project_path = "/kaggle/input/store-sales-time-series-forecasting/"

df_stores = pd.read_csv(project_path + "stores.csv")
df_oil = pd.read_csv(project_path + "oil.csv", parse_dates=["date"])
df_holidays = pd.read_csv(project_path + "holidays_events.csv", parse_dates=["date"])
df_transactions = pd.read_csv(project_path + "transactions.csv", parse_dates=["date"])

df = pd.read_csv(project_path + "train.csv", parse_dates=["date"])
df_test = pd.read_csv(project_path + "test.csv", parse_dates=["date"])

In [ ]:
FIRST_DATE = df["date"].min()
LAST_DATE = df_test["date"].max()
VAL_DATE = pd.to_datetime("2017-08-01")
ALL_DATES = pd.date_range(FIRST_DATE, LAST_DATE, freq='D')
CHRISTMAS_DAYS = [pd.to_datetime(f"201{n}-12-25") for n in [3,4,5,6]]
N_STORES = df["store_nbr"].nunique()
N_FAMILIES = df["family"].nunique()

In [ ]:
def RMSLE(y_true, y_pred):
    
    if isinstance(y_true, pd.Series):
        y_true = y_true.to_numpy()
    if isinstance(y_pred, pd.Series):
        y_pred = y_pred.to_numpy()
        
    return np.sqrt(np.mean((np.log(1 + y_pred) - np.log(1 + y_true))**2))

In [ ]:
df_stores.head(10)

In [ ]:
df_oil = df_oil.rename({"dcoilwtico": "oil_price"}, axis=1)
df_oil = df_oil.set_index("date").reindex(ALL_DATES).reset_index(names="date")
df_oil["oil_price"] = df_oil["oil_price"].interpolate(limit_direction="both")

df_oil

In [ ]:
df_oil.plot(x="date", y="oil_price", ylabel="oil price",
            title="Oil price over time", figsize=(15, 10));

In [ ]:
df_transactions

In [ ]:
df_holidays.head(10)

In [ ]:
df.head(10)

In [ ]:
df[df["date"].isin(CHRISTMAS_DAYS)]

In [ ]:
# Set date frequency to day to include all days (ie christmas days)

df = df.set_index(["date", "store_nbr", "family"])\
        .unstack(["store_nbr", "family"]).asfreq('D')\
        .stack(["store_nbr", "family"], future_stack=True)\
        .reset_index()

# Fill in missing christmas days

df["sales"] = df["sales"].fillna(0.0)
df["onpromotion"] = df["onpromotion"].fillna(0).astype(int)

df[df["date"].isin(CHRISTMAS_DAYS)]

In [ ]:
df = pd.merge(df, df_oil, on="date", how="left")

df

In [ ]:
df_test = pd.merge(df_test, df_oil, on="date", how="left")

df_test

In [ ]:
#from sktime.registry import all_estimators
#
#all_estimators(as_dataframe=True, 
#               filter_tags={'ignores-exogeneous-X': False, 
#                            "scitype:y": ["multivariate", "both"]})

In [ ]:
store_nbr = np.random.randint(N_STORES)

df_store = df[df["store_nbr"]==store_nbr]

df_store

In [ ]:
Y = df_store.pivot(index="date", columns="family", values="sales")
X = df_store.assign(family="PROMO_"+df_store["family"])\
            .pivot(index="date", columns="family", values="onpromotion")

Y

In [ ]:
Y.resample('W').mean()\
.plot(figsize=(15,10), ylabel="sales", title= "Sales per family");

In [ ]:
Y_train = Y.loc[:VAL_DATE].iloc[:-1].asfreq('D')
Y_val = Y.loc[VAL_DATE:].asfreq('D')
Y_train_diff = Y_train.diff(1).dropna()

X_train = X.loc[:VAL_DATE].iloc[:-1].asfreq('D')
X_val = X.loc[VAL_DATE:].asfreq('D')

Y_train

In [ ]:
%%time

scores = {}
for n_lags in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:
    
    model = VAR(maxlags=n_lags, trend='c')
    model.fit(Y_train_diff, X=X_train)
    fh = ForecastingHorizon(Y_val.index, freq='D')
    Y_pred_diff = model.predict(fh, X=X_val)
    Y_pred = (Y_train.iloc[-1] + Y_pred_diff.cumsum()).clip(lower=0)
    scores[n_lags] = RMSLE(Y_val.stack(), Y_pred.stack())

In [ ]:
pd.Series(scores).plot(xlabel="number of lags", title="RMSLE")
best_lags = min(scores, key=scores.get)
best_score = min(scores.values())
print("Best number of lags:", best_lags)
print("Best RMSLE score:", best_score)

In [ ]:
model = VAR(maxlags=best_lags, trend='c')
%time model.fit(Y_train_diff)

In [ ]:
fh = ForecastingHorizon(Y_val.index, freq='D')
#Y_pred = model.predict(fh, X=X_val).clip(lower=0)

Y_pred_diff = model.predict(fh, X=X_val)
Y_pred = (Y_train.iloc[-1] + Y_pred_diff.cumsum()).clip(lower=0)

Y_pred

In [ ]:
family = df_store["family"].unique()[np.random.randint(N_FAMILIES)]

plt.figure(figsize=(12,5))
plt.plot(Y_train[family], label="train data")
plt.plot(Y_val[family], label="new data")
plt.plot(Y_pred[family], label="predictions")
plt.xlim(pd.to_datetime("2017-05-01"), pd.to_datetime("2017-08-15"))
plt.title(f"Data and Predictions for family {family}")
plt.xlabel("date"); plt.ylabel("sales"); plt.legend();

In [ ]:
print(f"RMSLE for store {store_nbr} and family {family}:")
print(RMSLE(Y_val[family], Y_pred[family]))
print(f"RMSLE for store {store_nbr}:")
print(RMSLE(Y_val.stack(), Y_pred.stack()))

In [ ]:
pd.concat([Y_val.stack(), Y_pred.stack()], axis=1, keys=["sales_true", "sales_pred"])\
.reset_index().groupby("family")[["sales_true", "sales_pred"]]\
.apply(lambda group: RMSLE(group["sales_true"], group["sales_pred"]))\
.sort_values(ascending=False).head()

In [ ]:
%%time

df_scores = pd.DataFrame({"lags": [1]*N_STORES, 
                          "score": [0.0]*N_STORES}, 
                         index=range(1, N_STORES+1))
results = []

for store_nbr, df_store in df.groupby("store_nbr"):
    
    Y = df_store.pivot(index="date", columns="family", values="sales")
    X = df_store.assign(family="PROMO_"+df_store["family"])\
        .pivot(index="date", columns="family", values="onpromotion")
    
    Y_train = Y.loc[:VAL_DATE].iloc[:-1].asfreq('D')
    Y_val = Y.loc[VAL_DATE:].asfreq('D')
    Y_train_diff = Y_train.diff(1).dropna()
    
    df_val = df_store[df_store["date"] >= VAL_DATE]
    fh = ForecastingHorizon(Y_val.index, freq='D')
    
    X_train = X.loc[:VAL_DATE].iloc[:-1].asfreq('D')
    X_val = X.loc[VAL_DATE:].asfreq('D')
    
    scores = {}
    for n_lags in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:

        model = VAR(maxlags=n_lags, trend='c')
        model.fit(Y_train_diff, X=X_train)
        Y_pred_diff = model.predict(fh, X=X_val)
        Y_pred = (Y_train.iloc[-1] + Y_pred_diff.cumsum()).clip(lower=0)
        scores[n_lags] = RMSLE(Y_val.stack(), Y_pred.stack())
    
    best_lags = min(scores, key=scores.get)
    best_score = min(scores.values())
    df_scores.loc[store_nbr, "lags"] = best_lags
    df_scores.loc[store_nbr, "score"] = best_score
    
    model = VAR(maxlags=best_lags, trend='c')
    model.fit(Y_train_diff, X=X_train)
    
    Y_pred_diff = model.predict(fh, X=X_val)
    Y_pred = (Y_train.iloc[-1] + Y_pred_diff.cumsum()).clip(lower=0)
    
    df_pred = Y_pred.stack().reset_index(name="sales")
    df_pred["store_nbr"] = store_nbr
    
    df_result = pd.merge(df_val, df_pred, 
                         on=["date", "store_nbr", "family"], 
                         how="left", suffixes=("_true", "_pred"))
    
    results.append(df_result)

df_results = pd.concat(results, axis=0).sort_values("id").reset_index(drop=True)

In [ ]:
df_results

In [ ]:
RMSLE(df_results["sales_true"], df_results["sales_pred"])

In [ ]:
df_scores.sort_values("score", ascending=False).head()

In [ ]:
%%time

results = []

for store_nbr, df_store in df.groupby("store_nbr"):

    df_store_test = df_test[df_test["store_nbr"]==store_nbr]
    
    Y_train = df_store.pivot(index="date", columns="family", values="sales").asfreq('D')
    X_train = df_store.assign(family="PROMO_"+df_store["family"])\
        .pivot(index="date", columns="family", values="onpromotion").asfreq('D')
    X_test = df_store_test.assign(family="PROMO_"+df_store_test["family"])\
        .pivot(index="date", columns="family", values="onpromotion").asfreq('D')

    Y_train_diff = Y_train.diff(1).dropna()
    best_lags = df_scores.loc[store_nbr, "lags"]
    model = VAR(maxlags=best_lags, trend='c')
    model.fit(Y_train_diff, X=X_train)

    fh = ForecastingHorizon(X_test.index, freq='D')
    Y_pred_diff = model.predict(fh, X=X_test)
    Y_pred = (Y_train.iloc[-1] + Y_pred_diff.cumsum()).clip(lower=0)
    
    df_pred = Y_pred.stack().reset_index(name="sales")
    df_pred["store_nbr"] = store_nbr
    
    df_result = pd.merge(df_store_test, df_pred, 
                         on=["date", "store_nbr", "family"], 
                         how="left")
    
    results.append(df_result)

df_results = pd.concat(results, axis=0).sort_values("id").reset_index(drop=True)

In [ ]:
df_results

In [ ]:
df_submission = pd.read_csv(project_path + "sample_submission.csv")

df_submission["sales"] = df_results["sales"]

df_submission

In [ ]:
df_submission.to_csv("submission.csv", index=False)